In [7]:
pip install -q --upgrade google-genai pandas numpy matplotlib seaborn plotly streamlit scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [8]:
import os
import json
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from pathlib import Path
from getpass import getpass
from google import genai

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

Libraries imported successfully!


In [9]:
GEMINI_API_KEY = getpass("Enter Gemini API Key: ")

client = genai.Client(
    api_key=GEMINI_API_KEY
)

MODEL = "gemini-3.8-flash"

In [10]:
response = client.models.generate_content(
    model=MODEL,
    contents="Introduce yourself as MarketMind AI, an intelligent business analyst."
)

print(response.text)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [11]:
from pathlib import Path
import pandas as pd

# Define possible paths (adjust for your OS)
possible_paths = [
    Path.cwd() / "Users" / "User" / "Downloads" / "Marketmind ai",
    Path.cwd().parent / "Users" / "User" / "Downloads" / "Marketmind ai",
    Path.home() / "Downloads" / "Marketmind ai"  # Auto-detect Downloads folder
]

# Find the first existing path
DATA_PATH = next((p for p in possible_paths if p.exists()), None)

# Check before using DATA_PATH
if DATA_PATH is None:
    raise FileNotFoundError("No valid data path found. Please check your folder location.")


In [12]:
datasets = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "translation": "product_category_name_translation.csv"
}

data = {}

for name, filename in datasets.items():

    data[name] = pd.read_csv(DATA_PATH / filename)

    print(
        name,
        data[name].shape
    )

customers (99441, 5)
orders (99441, 8)
items (112650, 7)
payments (103886, 5)
reviews (99224, 7)
products (32951, 9)
sellers (3095, 4)
geolocation (1000163, 5)
translation (71, 2)


In [ ]:
date_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

for table, columns in date_columns.items():

    for column in columns:

        data[table][column] = pd.to_datetime(
            data[table][column],
            errors="coerce"
        )

print("Date conversion completed!")

Date conversion completed!


In [13]:
orders = data["orders"].copy()
items = data["items"].copy()
customers = data["customers"].copy()
products = data["products"].copy()
reviews = data["reviews"].copy()
translation = data["translation"].copy()

items["item_value"] = (
    items["price"] + items["freight_value"]
)

order_values = (
    items.groupby("order_id")
    .agg(
        order_value=("item_value", "sum"),
        units=("order_item_id", "count")
    )
    .reset_index()
)

sales = orders.merge(
    order_values,
    on="order_id",
    how="left"
)

sales["order_value"] = sales["order_value"].fillna(0)

sales["units"] = sales["units"].fillna(0)

print("Analytical dataset created:", sales.shape)

Analytical dataset created: (99441, 10)


In [14]:
def calculate_kpis():

    total_orders = sales["order_id"].nunique()

    delivered = sales[
        sales["order_status"] == "delivered"
    ]

    delivered_orders = delivered["order_id"].nunique()

    total_gmv = sales["order_value"].sum()

    delivered_gmv = delivered["order_value"].sum()

    delivered_aov = (
        delivered_gmv / delivered_orders
        if delivered_orders else 0
    )

    delivery_rate = (
        delivered_orders / total_orders * 100
        if total_orders else 0
    )

    return {
        "Total Orders": int(total_orders),
        "Total GMV": round(float(total_gmv), 2),
        "Delivered GMV": round(float(delivered_gmv), 2),
        "Delivered AOV": round(float(delivered_aov), 2),
        "Delivered Order Percentage": round(delivery_rate, 2)
    }

kpis = calculate_kpis()

kpis

{'Total Orders': 99441,
 'Total GMV': 15843553.24,
 'Delivered GMV': 15419773.75,
 'Delivered AOV': 159.83,
 'Delivered Order Percentage': 97.02}

In [15]:
category_data = items.merge(
    products[["product_id", "product_category_name"]],
    on="product_id",
    how="left"
)

category_data = category_data.merge(
    translation,
    on="product_category_name",
    how="left"
)

category_data["category"] = (
    category_data["product_category_name_english"]
    .fillna(category_data["product_category_name"])
)

category_analysis = (
    category_data.groupby("category")
    .agg(
        revenue=("item_value", "sum"),
        units_sold=("order_item_id", "count")
    )
    .reset_index()
)

category_analysis = category_analysis.sort_values(
    "revenue",
    ascending=False
)

category_analysis.head(10)

,category,revenue,units_sold
43,health_beauty,1441248.07,9670
72,watches_gifts,1305541.61,5991
7,bed_bath_table,1241681.72,11115
67,sports_leisure,1156656.48,8641
15,computers_accessories,1059272.40,7827
39,furniture_decor,902511.79,8334
49,housewares,778397.77,6964
20,cool_stuff,719329.95,3796
5,auto,685384.32,4235
42,garden_tools,584219.21,4347


In [16]:
monthly_sales = (
    sales.groupby(
        sales["order_purchase_timestamp"].dt.to_period("M")
    )
    .agg(
        revenue=("order_value", "sum"),
        orders=("order_id", "nunique")
    )
    .reset_index()
)

monthly_sales["month"] = (
    monthly_sales["order_purchase_timestamp"].astype(str)
)

monthly_sales["growth_percentage"] = (
    monthly_sales["revenue"].pct_change() * 100
)

monthly_sales

AttributeError: Can only use .dt accessor with datetimelike values

In [17]:
customer_orders = sales.merge(
    customers[["customer_id", "customer_unique_id", "customer_state"]],
    on="customer_id",
    how="left"
)

state_analysis = (
    customer_orders.groupby("customer_state")
    .agg(
        orders=("order_id", "nunique"),
        customers=("customer_unique_id", "nunique")
    )
    .reset_index()
    .sort_values("orders", ascending=False)
)

state_analysis.head(10)

,customer_state,orders,customers
25,SP,41746,40302
18,RJ,12852,12384
10,MG,11635,11259
22,RS,5466,5277
17,PR,5045,4882
23,SC,3637,3534
4,BA,3380,3277
6,DF,2140,2075
7,ES,2033,1964
8,GO,2020,1952


In [18]:
review_analysis = (
    reviews.groupby("review_score")
    .size()
    .reset_index(name="review_count")
)

average_rating = reviews["review_score"].mean()

print("Average Review Rating:", round(average_rating, 2))

display(review_analysis)

Average Review Rating: 4.09


,review_score,review_count
0,1,11424
1,2,3151
2,3,8179
3,4,19142
4,5,57328


In [19]:
delivery = sales[
    (sales["order_delivered_customer_date"].notna()) &
    (sales["order_estimated_delivery_date"].notna())
].copy()

delivery["delivery_days"] = (
    delivery["order_delivered_customer_date"] -
    delivery["order_purchase_timestamp"]
).dt.days

delivery["is_delayed"] = (
    delivery["order_delivered_customer_date"] >
    delivery["order_estimated_delivery_date"]
)

delivery_metrics = {
    "Average Delivery Days": round(
        delivery["delivery_days"].mean(), 2
    ),
    "Delayed Orders": int(
        delivery["is_delayed"].sum()
    ),
    "Delayed Order Percentage": round(
        delivery["is_delayed"].mean() * 100, 2
    )
}

delivery_metrics

TypeError: operation 'sub' not supported for dtype 'str' with dtype 'str'

In [20]:
def get_business_data(analysis_type):

    if analysis_type == "kpis":
        return calculate_kpis()

    elif analysis_type == "categories":
        return category_analysis.head(10).to_dict("records")

    elif analysis_type == "monthly":
        return monthly_sales.tail(12).to_dict("records")

    elif analysis_type == "customers":
        return state_analysis.head(10).to_dict("records")

    elif analysis_type == "reviews":
        return {
            "average_rating": round(float(average_rating), 2),
            "distribution": review_analysis.to_dict("records")
        }

    elif analysis_type == "delivery":
        return delivery_metrics

    else:
        return {
            "error": "Unsupported analysis type"
        }

In [21]:
def classify_question(question):

    prompt = f"""
    You are the question-routing engine for MarketMind AI.

    Classify the user's question into exactly one category:

    kpis
    categories
    monthly
    customers
    reviews
    delivery

    Return ONLY valid JSON in this format:

    {{
        "analysis_type": "category_name"
    }}

    User question:
    {question}
    """

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )

    result = json.loads(response.text)

    allowed = {
        "kpis",
        "categories",
        "monthly",
        "customers",
        "reviews",
        "delivery"
    }

    if result["analysis_type"] not in allowed:
        raise ValueError("Invalid analysis type returned")

    return result["analysis_type"]

In [22]:
def generate_insight(question, analysis_type, results):

    prompt = f"""
    You are MarketMind AI, an expert business intelligence analyst.

    Analyse the following actual business data.

    User question:
    {question}

    Analysis performed:
    {analysis_type}

    Actual analytical results:
    {json.dumps(results, default=str)}

    Instructions:

    1. Answer the user's question directly.
    2. Explain the important numerical findings.
    3. Identify meaningful business patterns.
    4. Explain possible business implications.
    5. Give practical recommendations.
    6. Never invent statistics or figures.
    7. Clearly distinguish observed facts from hypotheses.
    8. Use professional business language.

    Structure your answer using:

    - Direct Answer
    - Key Findings
    - Business Interpretation
    - Recommendations
    """

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )

    return response.text

In [23]:
def marketmind_analyst(question):

    print("Understanding your question...")

    analysis_type = classify_question(question)

    print("Selected analysis:", analysis_type)

    print("Retrieving actual business data...")

    results = get_business_data(analysis_type)

    print("Generating business insights...\n")

    answer = generate_insight(
        question,
        analysis_type,
        results
    )

    return answer

In [24]:
question = "Which product categories generate the highest revenue?"

answer = marketmind_analyst(question)

print(answer)

Understanding your question...


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [25]:
test_questions = [
    "What is the overall business performance?",
    "Which product categories generate the highest revenue?",
    "How are monthly sales changing?",
    "Which states have the highest customer concentration?",
    "How satisfied are our customers?",
    "Are our deliveries frequently delayed?"
]

for question in test_questions:

    print("\n" + "="*80)
    print("QUESTION:", question)
    print("="*80)

    answer = marketmind_analyst(question)

    print(answer)


QUESTION: What is the overall business performance?
Understanding your question...
Selected analysis: kpis
Retrieving actual business data...
Generating business insights...



ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [26]:
def marketmind_visual_analyst(question):

    analysis_type = classify_question(question)

    print("MarketMind AI Analysis:", analysis_type)

    # Generate textual business insights
    results = get_business_data(analysis_type)

    answer = generate_insight(
        question,
        analysis_type,
        results
    )

    print("\nBUSINESS INSIGHTS\n")
    print(answer)

    # Generate visualisation
    print("\nGENERATING VISUALISATION...\n")

    generate_chart(analysis_type)

In [27]:
marketmind_visual_analyst(
    "Which product categories generate the highest revenue?"
)

MarketMind AI Analysis: categories


KeyboardInterrupt: 

In [29]:
# Select delivered orders
delivered_sales = sales[
    sales["order_status"] == "delivered"
].copy()

# Ensure purchase timestamp is datetime
delivered_sales["order_purchase_timestamp"] = pd.to_datetime(
    delivered_sales["order_purchase_timestamp"],
    errors="coerce"
)

# Remove rows with invalid dates
delivered_sales = delivered_sales.dropna(
    subset=["order_purchase_timestamp"]
)

# Extract month as a timestamp directly
delivered_sales["month"] = (
    delivered_sales["order_purchase_timestamp"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

# Aggregate monthly sales
forecast_data = (
    delivered_sales.groupby("month")["order_value"]
    .sum()
    .reset_index()
)

forecast_data.columns = ["month", "sales"]

# Ensure month is datetime
forecast_data["month"] = pd.to_datetime(
    forecast_data["month"]
)

# Fill missing months
forecast_data = (
    forecast_data.set_index("month")
    .asfreq("MS", fill_value=0)
    .reset_index()
)

print("Forecast dataset prepared!")
display(forecast_data.head())

Forecast dataset prepared!


,month,sales
0,2016-09-01,143.46
1,2016-10-01,46490.66
2,2016-11-01,0.00
3,2016-12-01,19.62
4,2017-01-01,127482.37


In [30]:
forecast_data = forecast_data.sort_values("month").reset_index(drop=True)

# Historical sales features
forecast_data["lag_1"] = forecast_data["sales"].shift(1)
forecast_data["lag_2"] = forecast_data["sales"].shift(2)
forecast_data["lag_3"] = forecast_data["sales"].shift(3)

# Moving average of previous 3 months
forecast_data["rolling_mean_3"] = (
    forecast_data["sales"]
    .shift(1)
    .rolling(window=3)
    .mean()
)

# Seasonal feature
forecast_data["month_number"] = forecast_data["month"].dt.month

# Remove rows without historical information
model_data = forecast_data.dropna().copy()

display(model_data.head())

,month,sales,lag_1,lag_2,lag_3,rolling_mean_3,month_number
3,2016-12-01,19.62,0.00,46490.66,143.46,15544.706667,12
4,2017-01-01,127482.37,19.62,0.00,46490.66,15503.426667,1
5,2017-02-01,271239.32,127482.37,19.62,0.00,42500.663333,2
6,2017-03-01,414330.95,271239.32,127482.37,19.62,132913.770000,3
7,2017-04-01,390812.40,414330.95,271239.32,127482.37,271017.546667,4


In [32]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

features = [
    "lag_1",
    "lag_2",
    "lag_3",
    "rolling_mean_3",
    "month_number"
]

X = model_data[features]
y = model_data["sales"]

# Use the last 3 months for testing
X_train = X.iloc[:-3]
X_test = X.iloc[-3:]

y_train = y.iloc[:-3]
y_test = y.iloc[-3:]

rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=5,
    min_samples_leaf=2,
    random_state=42
)

rf_model.fit(X_train, y_train)

# Predictions
y_pred = rf_model.predict(X_test)

# Evaluation
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Model Evaluation")
print("-------------------------")
print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))

Model Evaluation
-------------------------
MAE: 66344.95
RMSE: 67389.56


In [33]:
comparison = pd.DataFrame({
    "Month": model_data["month"].iloc[-3:].values,
    "Actual Sales": y_test.values,
    "Predicted Sales": y_pred
})

comparison["Error"] = (
    comparison["Actual Sales"] -
    comparison["Predicted Sales"]
)

display(comparison.round(2))

,Month,Actual Sales,Predicted Sales,Error
0,2018-06-01,1011978.29,1079307.49,-67329.20
1,2018-07-01,1027807.28,1079209.39,-51402.11
2,2018-08-01,985491.64,1065795.18,-80303.54


In [34]:
history = forecast_data["sales"].tolist()

future_predictions = []

last_date = forecast_data["month"].max()

for i in range(1, 4):

    next_month = last_date + pd.DateOffset(months=i)

    features_next = pd.DataFrame([{
        "lag_1": history[-1],
        "lag_2": history[-2],
        "lag_3": history[-3],
        "rolling_mean_3": np.mean(history[-3:]),
        "month_number": next_month.month
    }])

    prediction = rf_model.predict(features_next)[0]

    prediction = max(0, prediction)

    future_predictions.append({
        "month": next_month,
        "predicted_sales": prediction
    })

    history.append(prediction)

future_forecast = pd.DataFrame(future_predictions)

display(future_forecast.round(2))

,month,predicted_sales
0,2018-09-01,1068445.05
1,2018-10-01,1059083.37
2,2018-11-01,1060383.77


In [35]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=forecast_data["month"],
    y=forecast_data["sales"],
    mode="lines+markers",
    name="Historical Sales"
))

fig.add_trace(go.Scatter(
    x=future_forecast["month"],
    y=future_forecast["predicted_sales"],
    mode="lines+markers",
    name="Predicted Sales",
    line=dict(dash="dash")
))

fig.update_layout(
    title="MarketMind AI – Three-Month Sales Forecast",
    xaxis_title="Month",
    yaxis_title="Delivered GMV",
    template="plotly_white"
)

fig.show()

In [36]:
forecast_data.to_csv(
    "historical_sales_forecast_data.csv",
    index=False
)

future_forecast.to_csv(
    "marketmind_sales_forecast.csv",
    index=False
)

print("Forecast results saved successfully!")

Forecast results saved successfully!


In [38]:
print(sales.columns.tolist())
print(data["customers"].columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_value', 'units']
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']


In [39]:
# Add customer_unique_id using customer_id
sales = sales.merge(
    data["customers"][["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
)

print("Customer ID columns added!")
print(sales[["customer_id", "customer_unique_id"]].head())

Customer ID columns added!
                        customer_id                customer_unique_id
0  9ef432eb6251297304e76186b10a928d  7c396fd4830fd04220f754e42b4e5bff
1  b0830fb4747a6c6d20dea0b8c802d7ef  af07308b275d755c9edb36a90c618231
2  41ce2a54c0b03bf3443c3d931a367089  3a653a41f6f9fc3d2a113cf8398680e8
3  f88197465ea7920adcdbec7375364d82  7c142cf63193a1473d2e66489a9ae977
4  8ab97904e6daea8866dbdbc4fb7aad2c  72632f0f9dd73dfee390c9b22eb56dd6


In [40]:
import pandas as pd
import numpy as np

# Select delivered orders
customer_orders = sales[
    sales["order_status"] == "delivered"
].copy()

# Ensure date format
customer_orders["order_purchase_timestamp"] = pd.to_datetime(
    customer_orders["order_purchase_timestamp"],
    errors="coerce"
)

# Remove missing customer IDs and dates
customer_orders = customer_orders.dropna(
    subset=["customer_unique_id", "order_purchase_timestamp"]
)

# Calculate RFM metrics
reference_date = (
    customer_orders["order_purchase_timestamp"].max()
    + pd.Timedelta(days=1)
)

rfm = customer_orders.groupby("customer_unique_id").agg(
    Recency=("order_purchase_timestamp",
             lambda x: (reference_date - x.max()).days),

    Frequency=("order_id", "nunique"),

    Monetary=("order_value", "sum")
).reset_index()

print("RFM dataset created successfully!")
print("Total customers:", len(rfm))

display(rfm.head())

RFM dataset created successfully!
Total customers: 93358


,customer_unique_id,Recency,Frequency,Monetary
0,0000366f3b9a7992bf8c76cfdf3221e2,112,1,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,115,1,27.19
2,0000f46a3911fa3c0805444483337064,537,1,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,321,1,43.62
4,0004aac84e0df4da2b147fca70cf8255,288,1,196.89


In [41]:
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

# Select RFM features
rfm_features = rfm[["Recency", "Frequency", "Monetary"]].copy()

# Ensure positive values for log transformation
rfm_features["Recency"] = np.log1p(rfm_features["Recency"])
rfm_features["Frequency"] = np.log1p(rfm_features["Frequency"])
rfm_features["Monetary"] = np.log1p(rfm_features["Monetary"])

# Standardize features
scaler = StandardScaler()

rfm_scaled = scaler.fit_transform(rfm_features)

print("RFM transformation and scaling completed!")
print("Scaled data shape:", rfm_scaled.shape)

RFM transformation and scaling completed!
Scaled data shape: (93358, 3)
